In [1]:
import numpy as np

input_data = np.load('initial_data/function_5/initial_inputs.npy')
print("Before:", input_data.shape)
new_point = np.array([[0.278167, 0.217734, 0.996929, 0.992772]])
input_data = np.vstack([input_data, new_point])
print("After:", input_data.shape)
print(input_data)


Before: (20, 4)
After: (21, 4)
[[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.278167   0.217734   0.996

In [ ]:
output_data = np.load('initial_data/function_5/initial_outputs.npy')
print("Before:", output_data.shape)
new_output = np.array([1552.6699801013826])
output_data = np.append(output_data, new_output)
print("After:", output_data.shape)
print(output_data)


(20,)
[6.44434399e+01 1.83013796e+01 1.12939795e-01 4.21089813e+00
 2.58370525e+02 7.84343889e+01 5.75715369e+01 1.09571876e+02
 8.84799176e+00 2.33223610e+02 2.44230883e+01 6.44201468e+01
 6.34767158e+01 7.97291299e+01 3.55806818e+02 1.08885962e+03
 2.88667516e+01 4.51815703e+01 4.31612757e+02 9.97233189e+00]


In [3]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C, Matern


In [4]:
n_dims = input_data.shape[1]
kernel = C(1.0) * Matern(length_scale=np.ones(n_dims), nu=2.5)

In [5]:
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, normalize_y=True)
gp.fit(input_data, output_data)


/Users/andriy/miniconda3/envs/appenv/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


GaussianProcessRegressor(kernel=1**2 * Matern(length_scale=[1, 1, 1, 1], nu=2.5),
                         n_restarts_optimizer=10, normalize_y=True)

In [ ]:
from scipy.stats import norm

def expected_improvement(X_candidates, gp, y_best, xi=0.01):
    mu, sigma = gp.predict(X_candidates, return_std=True)
    Z = (mu - y_best - xi) / (sigma + 1e-9)
    ei = (mu - y_best - xi) * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei

# Generate candidate points and find best
candidates = np.random.rand(1000, n_dims)
ei_scores = expected_improvement(candidates, gp, y_best=output_data.max())
best_next = candidates[np.argmax(ei_scores)]
print(f"Suggested next input: {'-'.join(f'{x:.6f}' for x in best_next)}")

Suggested next input: [0.27816776 0.2177346  0.99692903 0.99277256]


In [10]:
mean, std = gp.predict([best_next], return_std=True)

print(f"Predicted mean: {mean[0]:.6e}")
print(f"Uncertainty (std): {std[0]:.6e}")

Predicted mean: 5.928535e-01
Uncertainty (std): 1.303603e-01
